In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../Dataset/diabetes.csv')

In [3]:
x = df.iloc[:,0:-1]
y = df.iloc[:,-1]

In [4]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state= 42)

In [5]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [6]:
import tensorflow
import keras
from keras.models import Sequential
from keras.layers import Dense,Dropout

In [7]:
import keras_tuner as kt

In [16]:
def build_model(hp):
    model = Sequential()

    flag = 0

    for i in range(hp.Int('num layers', min_value = 1, max_value = 10)):
        if flag == 0:
            model.add(Dense( hp.Int(str(i)+' layer units', min_value = 8, max_value = 128), 
                            activation = hp.Choice('activation', values = ['sigmoid', 'relu', 'tanh']),
                            input_dim = 8))
            model.add(Dropout(hp.Choice('Dropout'+str(i), values =[0.1, 0.2, 0.3, 0.4, 0.5])))
            flag = 1
        else:
            model.add(Dense( hp.Int(str(i)+' layer units', min_value = 8, max_value = 128), 
                            activation = hp.Choice('activation', values = ['sigmoid', 'relu', 'tanh']),
                            ))
            model.add(Dropout(hp.Choice('Dropout'+str(i), values =[0.1, 0.2, 0.3, 0.4, 0.5])))

    model.add(Dense(1, activation ='sigmoid'))

    model.compile(loss = 'binary_crossentropy', optimizer = hp.Choice('optimizer', values = ['adam', 'rmsprop', 'adadelta']), metrics = ['accuracy'])

    return model

In [17]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_loss',
                        max_trials= 5)

/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
tuner.search(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Trial 5 Complete [00h 00m 03s]
val_loss: 0.727751612663269

Best val_loss So Far: 0.5700267553329468
Total elapsed time: 00h 00m 14s


In [19]:
tuner.get_best_hyperparameters()[0].values

{'num layers': 8,
 '0 layer units': 94,
 'activation': 'relu',
 'Dropout0': 0.5,
 'optimizer': 'rmsprop',
 '1 layer units': 8,
 'Dropout1': 0.1,
 '2 layer units': 8,
 'Dropout2': 0.1,
 '3 layer units': 8,
 'Dropout3': 0.1,
 '4 layer units': 8,
 'Dropout4': 0.1,
 '5 layer units': 8,
 'Dropout5': 0.1,
 '6 layer units': 8,
 'Dropout6': 0.1,
 '7 layer units': 8,
 'Dropout7': 0.1}

In [20]:
model = tuner.get_best_models(num_models = 1)[0]

/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 20 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 94)             │           846 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 94)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,047 (8.00 KB)

 Trainable params: 2,047 (8.00 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.fit(x_train, y_train, epochs = 100, initial_epoch=6, validation_data = (x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.6531 - loss: 0.5714 - val_accuracy: 0.6429 - val_loss: 0.5534
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6531 - loss: 0.5580 - val_accuracy: 0.6429 - val_loss: 0.5483
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6531 - loss: 0.5472 - val_accuracy: 0.6429 - val_loss: 0.5446
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6531 - loss: 0.5586 - val_accuracy: 0.6429 - val_loss: 0.5441
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6531 - loss: 0.5253 - val_accuracy: 0.6429 - val_loss: 0.5418
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6531 - loss: 0.5275 - val_accuracy: 0.6429 - val_loss: 0.5413
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7134 - loss: 0.5170 - val_accuracy: 0.7468 - val_loss: 0.5436
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7313 - loss: 0.5308 - val_accuracy: 0.73

In [26]:
y_pred = model.predict(x_test)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
